# Benchmarks

How the cost of the DQVP pipeline grows with the size of the policy and of the data. The notebook runs the measurement functions of `bench_planning.py` and `bench_execution.py` (in `tools/dqvp/operationalization-pipeline`) and shows their results. The outputs saved here are the run reported in the README.

To run it again: install `tools/dqvp/requirements.txt` and Jupyter, put the [EYE reasoner](https://github.com/eyereasoner/eye) on `PATH`, and run all cells. It takes a few minutes and writes its scratch files, including a 5,000,000-record CSV, to the pipeline's `artifacts/` folder.

In [1]:
import os, platform, statistics, subprocess, sys, time
from datetime import date
from pathlib import Path

import pandas as pd

PIPELINE = Path("../tools/dqvp/operationalization-pipeline").resolve()
os.chdir(PIPELINE)
sys.path.insert(0, str(PIPELINE))

import duckdb, rdflib
import codegen as C
import dqr_to_dqvp as M
import bench_planning as BP
import bench_execution as BE

duckdb.sql("SET enable_progress_bar = false")   # no progress widgets in the saved outputs
pd.set_option("display.max_colwidth", None)

def proc_field(path, key):
    try:
        for line in Path(path).read_text().splitlines():
            if line.startswith(key):
                return line.split(":", 1)[1].strip()
    except OSError:
        return None

eye = subprocess.run(["eye", "--version"], capture_output=True, text=True)
mem = proc_field("/proc/meminfo", "MemTotal")
pd.Series({
    "Date": date.today().isoformat(),
    "CPU": f"{proc_field('/proc/cpuinfo', 'model name') or platform.processor()}, {os.cpu_count()} threads",
    "Memory available": f"{int(mem.split()[0]) / 2**20:.1f} GiB" if mem else "unknown",
    "System": platform.platform(),
    "Python": platform.python_version(),
    "DuckDB": duckdb.__version__,
    "rdflib": rdflib.__version__,
    "EYE": (eye.stdout + eye.stderr).strip().splitlines()[0],
}).to_frame("Machine")

,Machine
Date,2026-10-23
CPU,"AMD Ryzen 5 PRO 215 w/ Radeon 740M Graphics, 12 threads"
Memory available,7.1 GiB
System,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
Python,3.12.3
DuckDB,1.5.6
rdflib,7.6.0
EYE,EYE v11.24.4 (2026-05-12)


## Compilation as the policy grows

Each synthetic policy has *n* constraints on the Patient Summary columns, cycling through non-null, unique and null rates. Planning derives the policy checker with the reasoner; code generation writes the checks for every engine. Each size is the median of seven runs, and planning includes the reasoner's start-up of about 0.3 s. A cost that is linear in the number of checks doubles when the policy doubles (×2.0); a quadratic one would quadruple.

In [2]:
REPEATS = 7
rows = []
for n in (32, 64, 128, 256, 512):
    runs = [BP.bench(n) for _ in range(REPEATS)]
    plan_s = statistics.median(r[0] for r in runs)
    gen_s = statistics.median(r[1] for r in runs)
    rows.append({"Checks": runs[0][3], "Operations": runs[0][2],
                 "Planning (s)": plan_s, "Code generation (s)": gen_s,
                 "Per check (ms)": 1000 * (plan_s + gen_s) / runs[0][3]})
compilation = pd.DataFrame(rows)
total = compilation["Planning (s)"] + compilation["Code generation (s)"]
compilation["× per doubling"] = total / total.shift()
compilation.round({"Planning (s)": 2, "Code generation (s)": 2, "Per check (ms)": 1, "× per doubling": 2})

,Checks,Operations,Planning (s),Code generation (s),Per check (ms),× per doubling
0,32,129,0.73,0.13,26.9,NaN
1,64,257,1.21,0.28,23.3,1.73
2,128,513,2.17,0.54,21.2,1.82
3,256,1025,4.14,1.13,20.6,1.94
4,512,2049,8.01,2.13,19.8,1.93


## Execution as the data grow

The three mapped EHDS DQRs (DQR1EH to DQR3EH) over the 500-record synthetic Patient Summary, replicated up to 5,000,000 records. Each copy of a record gets its own patient identifier, so a uniqueness check sees as many distinct values as a real table would; every other proportion is kept, so the measured values must not change with size. Loading the CSV and running the checks are timed separately, each the median of five runs.

In [3]:
REPEATS = 5
policy = M.ART / "policy_from_dqr.ttl"
saved_policy = policy.read_text() if policy.exists() else None
policy.write_text(M.build_policy([r for r in M.load_requirements() if r["id"] in ("DQR1EH", "DQR2EH", "DQR3EH")]))

rows, reference = [], None
for n in (500, 5_000, 50_000, 500_000, 5_000_000):
    ehds_plan = C.build_plan(BE.descriptor(BE.make_csv(n)), policy=policy)
    t_load = BE.median_load(ehds_plan, REPEATS)
    ehds_con, ehds_tables, _ = BE.load(ehds_plan)
    t_checks, values = BE.median_checks(ehds_plan, ehds_con, ehds_tables, REPEATS)
    assert reference is None or values == reference, (n, values, reference)
    reference = values
    rows.append({"Records": f"{n:,}", "Load (s)": t_load, "Three checks (s)": t_checks})
    if n < 5_000_000:
        ehds_con.close()
print("Measured values, the same at every size:", reference)
pd.DataFrame(rows).round(4)

Measured values, the same at every size: {'c-completeness': 98.0, 'c-compliance': 98.1964, 'c-consistency': 100.0}


,Records,Load (s),Three checks (s)
0,500,0.0840,0.0079
1,"5,000",0.0713,0.0221
2,"50,000",0.2059,0.1139
3,"500,000",0.4703,0.1358
4,"5,000,000",2.3222,0.5775


## Execution as the checks grow

Synthetic policies of 3 to 96 checks, like those of the compilation benchmark, over the 5,000,000-record table, loaded once. Each is the median of five runs.

In [4]:
csv = BE.descriptor(BE.make_csv(5_000_000))
rows, con = [], None
for k in (3, 12, 48, 96):
    pol = BE.BENCH / f"policy_{k}.ttl"
    pol.write_text(BP.synth_policy(k))
    plan = C.build_plan(csv, policy=pol)
    if con is None:
        con, tables, _ = BE.load(plan)
    if k == 12:
        plan12 = plan
    t, values = BE.median_checks(plan, con, tables, REPEATS)
    rows.append({"Checks": len(values), "Time (s)": t, "Per check (ms)": 1000 * t / len(values)})
    pol.unlink()
pd.DataFrame(rows).round({"Time (s)": 3, "Per check (ms)": 1})

,Checks,Time (s),Per check (ms)
0,3,0.071,23.8
1,12,0.350,29.2
2,48,1.119,23.3
3,96,2.075,21.6


## Single checks over 5,000,000 records

Each check of the three EHDS DQRs and of the 12-check synthetic policy, timed on its own (median of five runs). The cost of a check is the cost of its query, so it depends on the metric: counting nulls is cheap, counting distinct identifiers and testing each value against the 249 ISO 3166 codes are not.

In [5]:
def single_checks(plan, con, tables):
    out = []
    for c in C.checks(plan):
        sql = C._measure_sql(plan, c, C.context_of(plan, c), tables[C.workflow_of_check(plan, c)])
        times = []
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            con.execute(sql).fetchone()
            times.append(time.perf_counter() - t0)
        column = plan.value(c, C.DQVP.targetAttribute)
        out.append({"Check": C.local(c), "Metric": C.local(plan.value(c, C.ODRL.leftOperand)),
                    "Column": C.local(column) if column is not None else "", "Time (ms)": 1000 * statistics.median(times)})
    return out

single = pd.DataFrame(single_checks(ehds_plan, ehds_con, ehds_tables) + single_checks(plan12, con, tables))
ehds_con.close(); con.close()
if saved_policy is not None:
    policy.write_text(saved_policy)          # leave the pipeline's EHDS policy as it was
single.round({"Time (ms)": 1})

,Check,Metric,Column,Time (ms)
0,c-completeness,NonNullRate,nationalHealthcarePatientID,6.8
1,c-compliance,ReferenceConformance,countryOfAffiliation,589.0
2,c-consistency,NullRate,pregnancyHistory,12.0
3,b0,NonNullRate,nationalHealthcarePatientID,7.5
4,b1,UniqueRate,familyName,42.4
5,b10,UniqueRate,nationalHealthcarePatientID,199.1
6,b11,NullRate,familyName,8.1
7,b2,NullRate,givenName,8.4
8,b3,NonNullRate,dateOfBirth,2.1
9,b4,UniqueRate,gender,36.7
